# 07. 'agent'의 인접 연어(collocation) — 어떤 말과 함께 쓰이는가, 세계 전체 vs 한국어·중국어·일본어 집합 (2023-01 ~ 2026-09)

- **목적**(D37): LLM 에이전트 논문에서 'agent'가 바로 앞·뒤에 어떤 말과 함께 쓰이는지(multi-agent, llm agent, embodied agent / agent system, agent framework)를 문서 빈도로 세고, 그 경향이 세계 전체와 대상 언어 집합(D36 라벨 반영) 사이에, 그리고 연도 사이에 어떻게 다른지 본다. 토픽 모델과 달리 결정론적이고 재실행해도 같다.
- **입력**: `analysis/output/cache/agent_colloc_2023_2026q3.jsonl`(12_agent_collocations.py: spaCy 표제어·합성어 단위, 정본 `verification/agent_colloc_extract_*.json`), `preprocessing/output/corpus_llm_agent_all_2023_2026q3.csv`(연도·대상 언어 열)
- **자리(slot)**: `앞 말` = 접두(multi-agent의 multi) + 바로 앞 수식어(llm, autonomous, llm-based) / `뒤 말` = 바로 다음 명사(system, framework) / `agentic 뒤 말`
- **주의**
  - 문서 빈도(document frequency): 한 논문에 여러 번 나와도 1. 분모는 집합의 논문 수(세계 20,107 / 언어 집합 크기)
  - 수식어가 될 수 있는 말(명사·형용사·분사·합성어)만 센다. 'we propose agents', 'agents increasingly'의 동사·부사는 연어가 아니다
  - 'agent-based'처럼 agent가 접두인 합성어, 'agentic'은 머리(head)가 아니다. agentic은 뒤 말만 따로 본다
  - 2026은 1~9월. 언어 집합은 작으므로(한국어 25, 일본어 18) 비율보다 편수를 함께 읽는다


In [ ]:
import glob, json, re, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "verification").exists())   # 노트북은 analysis/notebooks/ 에 있다
DATA = ROOT / "preprocessing/output"                                   # 입력: 전처리 산출(정제 CSV)
VER = ROOT / "verification"                                            # 수치의 정본 JSON
FIG = ROOT / "analysis/output/figures"; FIG.mkdir(parents=True, exist_ok=True)   # 출력: 노트북 그림. 원고 그림은 paper/figures/
TAB = ROOT / "analysis/output/tables"; TAB.mkdir(parents=True, exist_ok=True)    # 출력: 노트북 표
CACHE = ROOT / "analysis/output/cache"; CACHE.mkdir(parents=True, exist_ok=True) # 출력: 재실행용 캐시(git 제외)
TODAY = pd.Timestamp.today().strftime("%Y%m%d")
SUFFIX = "2023_2026q3"   # D30: 분석 기간 2023-01 ~ 2026-09 말뭉치

try:
    sys.path.insert(0, str(ROOT / "analysis/scripts"))
    import figure_style; figure_style.apply_style(); C = figure_style.COLORS; CYC = figure_style.COLOR_CYCLE
except Exception:
    C = {"green": "#46B07B", "orange": "#FF9E02", "teal": "#3A9E8F", "coral": "#E87B4A", "green_dark": "#2D7A53", "orange_dark": "#C47800", "green_light": "#8DD4AA", "orange_light": "#FFD080", "green_pale": "#C8EBDA", "orange_pale": "#FFECC0", "gray": "#7F8C8D", "gray_light": "#BDC3C7", "gray_dark": "#4A4A4A", "text": "#333333"}; CYC = [C["green"], C["orange"], C["teal"], C["coral"]]
    plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

LANGS = ["ko", "zh", "ja"]
LANG_NAME = {"ko": "Korean", "zh": "Chinese", "ja": "Japanese"}
YEARS = [2023, 2024, 2025, 2026]          # 2026은 1~9월(D30)
MIN_N = 10   # 투영 집합의 토픽별 건수가 이보다 작으면 lift를 보고하지 않는다

allc = pd.read_csv(DATA / f"corpus_llm_agent_all_{SUFFIX}.csv", dtype={"id": str})
cjk_signal = pd.read_csv(DATA / f"corpus_llm_agent_cjk_lang_{SUFFIX}.csv", dtype={"id": str})   # 키워드 신호 257편 + 사람 라벨(D36)
counts = json.loads(Path(sorted(glob.glob(str(VER / "corpus_counts_*.json")))[-1]).read_text())
YEAR_BASIS = counts["definitions"]["year_basis"]   # D31: "id" = arXiv id 접두(최초 제출 월)
LABEL_TARGET = "그 언어의 텍스트를 실제 연구 대상으로 다룸"; LABEL_MULTI = "다국어 실험의 한 언어로 포함"
cjk = cjk_signal[cjk_signal.cjk_lang].copy()       # 최종 대상 언어 집합 = 신호 ∧ 라벨 '실제 연구 대상'(D36)

# 정본 대조: 노트북이 읽은 자료가 정본 JSON과 같은지 확인한다
assert YEAR_BASIS == "id", "연도 기준이 D31(id 접두)이 아니다"
assert len(allc) == counts["pipeline"]["llm_agent_all"], "전체 건수가 정본과 다르다"
assert len(cjk_signal) == counts["pipeline"]["llm_agent_cjk_signal"], "대상 언어 신호 건수가 정본과 다르다"
assert len(cjk) == counts["pipeline"]["llm_agent_cjk_lang"], "대상 언어 집합 건수가 정본과 다르다"
assert (cjk_signal.lang_label != "").all(), "라벨 없는 신호 논문이 있다"
assert allc.groupby("year").size().to_dict() == {int(k): v for k, v in counts["llm_agent_by_year"].items()}
assert set(cjk.id) <= set(allc.id) and allc.id.is_unique
for l in LANGS:
    allc[f"lang_{l}"] = allc[f"lang_{l}"].astype(bool); allc[f"lang_{l}_signal"] = allc[f"lang_{l}_signal"].astype(bool)
assert {l: int(allc[f"lang_{l}"].sum()) for l in LANGS} == counts["cjk_lang_by_lang_total"]
N_ALL_BY_YEAR = {int(k): v for k, v in counts["pipeline"]["unique_by_year"].items()}   # 분모: 다섯 cs 분류의 고유 논문 수
print("정본 대조 통과: LLM 에이전트", len(allc), "| 한중일어 신호", len(cjk_signal), "| 대상 언어 집합(사람 라벨)", len(cjk), "| 연도 기준", YEAR_BASIS, "| 기간", counts["definitions"]["period"][:24])

## 1. 자료 읽기 — 연어 캐시와 말뭉치 결합

- 계산: 캐시의 논문별 연어를 자리별 집합으로 바꿔 긴 표(id, slot, word)로 만든다. `앞 말`은 prefix와 left1을 합친다
- 정본 대조: 캐시의 논문 수 = 추출 정본의 n_docs = 말뭉치 전체


In [ ]:
ext = json.loads(Path(sorted(glob.glob(str(VER / "agent_colloc_extract_*.json")))[-1]).read_text())
recs = [json.loads(l) for l in open(CACHE / f"agent_colloc_{SUFFIX}.jsonl")]
assert len(recs) == ext["n_docs"] == len(allc), "캐시·정본·말뭉치의 논문 수가 다르다"
rows = []
for r in recs:
    for w in set(r["prefix"]) | set(r["left1"]): rows.append((r["id"], "before", w))
    for w in set(r["right1"]): rows.append((r["id"], "after", w))
    for w in set(r["agentic_right1"]): rows.append((r["id"], "agentic_after", w))
col = pd.DataFrame(rows, columns=["id", "slot", "word"]).drop_duplicates()
col = col.merge(allc[["id", "year"] + [f"lang_{l}" for l in LANGS]], on="id", how="left")
n_head = sum(r["n_head"] > 0 for r in recs)
N = {"world": len(allc), **{l: int(allc[f"lang_{l}"].sum()) for l in LANGS}}
print(f"논문 {len(allc):,}편 중 agent 머리 출현 {n_head:,}편({100*n_head/len(allc):.1f}%), 머리 출현 {ext['n_head_occurrences']:,}회 | 연어 행 {len(col):,} | 집합 크기 {N}")

## 2. 세계 전체 — agent 앞 말·뒤 말 상위 30과 연도 추이

- 계산: 자리별 단어의 문서 빈도와 비율(분모 20,107), 연도별 비율(분모 그 해 논문 수), 네 점 직선 기울기(pp/년)
- 출력: `analysis/output/tables/agent_colloc_before_world.csv`, `agent_colloc_after_world.csv`, `agent_colloc_agentic_after_world.csv`
- 읽는 법: 'multi'는 multi-agent, 'llm'은 llm agent(s), 'llm-based'는 llm-based agent(s). 기울기 +면 그 쓰임이 늘고 있다


In [ ]:
n_year = allc.groupby("year").size().reindex(YEARS)
def world_table(slot, top=30):
    g = col[col.slot == slot]
    t = g.groupby("word").id.nunique().rename("n").to_frame(); t["pct"] = (100 * t.n / N["world"]).round(2)
    for y in YEARS:
        t[f"pct_{y}"] = (100 * g[g.year == y].groupby("word").id.nunique().reindex(t.index).fillna(0) / n_year[y]).round(2)
    t["slope_pp_per_year"] = [round(float(np.polyfit(YEARS, t.loc[w, [f"pct_{y}" for y in YEARS]].values.astype(float), 1)[0]), 2) for w in t.index]
    return t.sort_values("n", ascending=False).head(top)
W = {s: world_table(s) for s in ("before", "after", "agentic_after")}
for s, t in W.items(): t.to_csv(TAB / f"agent_colloc_{s}_world.csv")
print("agent 앞 말(세계, 문서 %):", ", ".join(f"{w}({r.pct:.1f}%, {r.slope_pp_per_year:+.1f})" for w, r in W["before"].head(20).iterrows()))
print("agent 뒤 말(세계, 문서 %):", ", ".join(f"{w}({r.pct:.1f}%, {r.slope_pp_per_year:+.1f})" for w, r in W["after"].head(20).iterrows()))
print("agentic 뒤 말(세계, 문서 %):", ", ".join(f"{w}({r.pct:.1f}%, {r.slope_pp_per_year:+.1f})" for w, r in W["agentic_after"].head(12).iterrows()))
W["before"]

## 3. 세계 vs 한국어·중국어·일본어 집합 — 같은 단어의 관측 편수와 기대 편수

- 계산: 세계 상위 단어마다 언어 집합의 관측 편수(문서 빈도)와 기대 편수(세계 비율 × 집합 크기). 관측/기대가 1.5 초과면 많음, 0.5 미만이면 적음(7b절 LDA 표와 같은 기준). 집합이 작으므로 기대 1 미만인 칸은 판정하지 않는다
- 그리고 언어 집합에만 두드러지는 단어: 집합 안 3편 이상이고 세계 비율의 3배 이상
- 출력: `analysis/output/tables/agent_colloc_before_lang.csv`, `agent_colloc_after_lang.csv`, `agent_colloc_lang_specific.csv`, `analysis/output/figures/fig8a_agent_colloc_before_lang.png`


In [ ]:
def lang_table(slot, top=30):
    t = W[slot].copy()[["n", "pct"]].rename(columns={"n": "world_n", "pct": "world_pct"})
    g = col[col.slot == slot]
    for l in LANGS:
        obs = g[g[f"lang_{l}"]].groupby("word").id.nunique().reindex(t.index).fillna(0).astype(int)
        t[f"{l}_obs"] = obs; t[f"{l}_pct"] = (100 * obs / N[l]).round(1); t[f"{l}_exp"] = (t.world_pct / 100 * N[l]).round(1)
        ratio = obs / t[f"{l}_exp"].replace(0, np.nan)
        t[f"{l}_gap"] = np.where(t[f"{l}_exp"] < 1, "n/a", np.where(ratio > 1.5, "over", np.where(ratio < 0.5, "under", "near")))
    return t
L = {s: lang_table(s) for s in ("before", "after")}
for s, t in L.items(): t.to_csv(TAB / f"agent_colloc_{s}_lang.csv")
def md_cell(r, l): return f"{int(r[f'{l}_obs'])} ({r[f'{l}_pct']:.0f}%) · 기대 {r[f'{l}_exp']:.1f}" + {"over": " · 많음", "under": " · 적음"}.get(r[f"{l}_gap"], "")
md = f"| 자리 | 단어 | 세계 (n={N['world']:,}) | " + " | ".join(f"{LANG_NAME[l]} (n={N[l]})" for l in LANGS) + " |\n|---|---|---|---|---|---|\n"
for s, name in (("before", "agent 앞"), ("after", "agent 뒤")):
    for w, r in L[s].head(15).iterrows():
        md += f"| {name} | {w} | {int(r.world_n):,} ({r.world_pct:.1f}%) | " + " | ".join(md_cell(r, l) for l in LANGS) + " |\n"
(TAB / "agent_colloc_world_vs_lang.md").write_text("문서 빈도(논문 수)와 집합 내 비율. 기대 = 세계 비율 × 집합 크기. 많음 = 기대의 1.5배 초과, 적음 = 절반 미만(기대 1 미만은 판정 안 함).\n\n" + md)
# 언어 집합에만 두드러지는 단어
spec_rows = []
for s in ("before", "after"):
    g = col[col.slot == s]; wp = (g.groupby("word").id.nunique() / N["world"])
    for l in LANGS:
        lp = g[g[f"lang_{l}"]].groupby("word").id.nunique()
        for w, n in lp[lp >= 3].items():
            if n / N[l] >= 3 * wp.get(w, 0):
                spec_rows.append({"slot": s, "lang": LANG_NAME[l], "word": w, "lang_n": int(n), "lang_pct": round(100 * n / N[l], 1), "world_pct": round(100 * wp.get(w, 0), 2)})
spec = pd.DataFrame(spec_rows).sort_values(["lang", "slot", "lang_n"], ascending=[True, True, False]); spec.to_csv(TAB / "agent_colloc_lang_specific.csv", index=False)
print("언어 집합에 두드러지는 연어(3편 이상, 세계 비율의 3배 이상):"); print(spec.to_string(index=False) if len(spec) else "(없음)")
display(L["before"].head(15))
fig, ax = plt.subplots(figsize=(8, 5.5)); t = L["before"].head(20); y = np.arange(len(t))
ax.barh(y, t.world_pct, color="#B0B7C3", label=f"world (n={N['world']:,})")
for l, mk, c in zip(LANGS, ["o", "s", "^"], ["#C0392B", "#2E86C1", "#27AE60"]): ax.scatter(t[f"{l}_pct"], y, marker=mk, color=c, label=f"{LANG_NAME[l]} (n={N[l]})", zorder=3)
ax.set_yticks(y); ax.set_yticklabels(t.index); ax.invert_yaxis(); ax.set_xlabel("% of papers in the set (document frequency)"); ax.set_title("Words immediately before 'agent(s)': world vs target-language sets"); ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "fig8a_agent_colloc_before_lang.png", dpi=160); plt.show()

## 3b. 언어 집합 칸의 이항 검정과 llm 계열 합산 (리뷰 2026-10-14)

- 계산: 세계 비율을 귀무 비율로 두고 집합 관측 편수의 양측 이항 검정 p(보정 없음, 기술 통계). 앞 말·뒤 말 세계 상위 30 × 언어 3 = 180칸 + 언어 특이어.
- llm 계열 합산: 앞 말이 llm, llm-* (llm-based, llm-driven, llm-powered 등), 'language model', 'language model-based' 가운데 하나라도 있는 논문 수(논문 단위 합집합)와 세계 비율 기준 기대·이항 p.
- 출력: `analysis/output/tables/agent_colloc_binomial.csv`


In [ ]:
from scipy.stats import binomtest
rows_b = []
for s_ in ("before", "after"):
    g = col[col.slot == s_]; wp = g.groupby("word").id.nunique() / N["world"]
    words_b = set(W[s_].index) | set(spec[spec.slot == s_].word)
    for w in words_b:
        for l in LANGS:
            n_l = int(g[g[f"lang_{l}"] & (g.word == w)].id.nunique()); p0 = float(wp.get(w, 0))
            rows_b.append({"slot": s_, "word": w, "lang": l, "obs": n_l, "n": N[l], "world_pct": round(100 * p0, 2), "exp": round(p0 * N[l], 1), "lang_pct": round(100 * n_l / N[l], 1), "p_binom": round(binomtest(n_l, N[l], p0).pvalue, 4) if p0 > 0 else None})
BT = pd.DataFrame(rows_b); BT.to_csv(TAB / "agent_colloc_binomial.csv", index=False)
sigB = BT[(BT.p_binom.notna()) & (BT.p_binom < 0.05)].sort_values(["lang", "slot", "p_binom"])
print("이항 검정 p<0.05인 칸:"); print(sigB[["lang", "slot", "word", "obs", "n", "exp", "world_pct", "lang_pct", "p_binom"]].to_string(index=False))
# llm 계열 합산(논문 단위)
LLM_RE_W = re.compile(r"^(llm|llm-.*|language model|language model-based|large language model)$")
llm_fam = col[(col.slot == "before") & col.word.str.match(LLM_RE_W)]
llm_union = {"world": int(llm_fam.id.nunique())}
for l in LANGS: llm_union[l] = int(llm_fam[llm_fam[f"lang_{l}"]].id.nunique())
p0 = llm_union["world"] / N["world"]
llm_union_stats = {l: {"obs": llm_union[l], "exp": round(p0 * N[l], 1), "p_binom": round(binomtest(llm_union[l], N[l], p0).pvalue, 4)} for l in LANGS}
print(f"\nllm 계열 합산: 세계 {llm_union['world']:,}편({100*p0:.1f}%)", llm_union_stats)

## 4. 연도 추이 — 세계 상위 앞 말·뒤 말의 비율 변화

- 계산: 2절의 연도별 비율을 선으로. 분모는 그 해 논문 수(2026은 1~9월)
- 출력: `analysis/output/figures/fig8b_agent_colloc_trend.png`


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for ax, s, title in zip(axes, ("before", "after"), ("before 'agent(s)'", "after 'agent(s)'")):
    t = W[s].head(10)
    for w, r in t.iterrows(): ax.plot(YEARS, [r[f"pct_{y}"] for y in YEARS], marker="o", label=w)
    ax.set_xticks(YEARS); ax.set_xticklabels([str(y) if y != 2026 else "2026\n(Jan-Sep)" for y in YEARS]); ax.set_ylabel("% of papers that year"); ax.set_title(f"Top 10 words {title}"); ax.legend(fontsize=7, frameon=False, ncol=2)
plt.tight_layout(); plt.savefig(FIG / "fig8b_agent_colloc_trend.png", dpi=160); plt.show()
rising = W["before"][W["before"].slope_pp_per_year >= 0.5].index.tolist(); falling = W["before"][W["before"].slope_pp_per_year <= -0.5].index.tolist()
print("앞 말 오름(+0.5pp/년 이상):", rising); print("앞 말 내림(−0.5pp/년 이하):", falling)

## 5. 정본 저장

- 저장: 세계 상위 표, 언어 집합 관측·기대, 언어 특이 연어, 집합 크기 → `verification/agent_colloc_YYYYMMDD.json`


In [ ]:
out = {"generated": TODAY, "notebook": "analysis/notebooks/07_agent_collocations.ipynb", "period": counts["definitions"]["period"], "extract": ext,
       "set_sizes": N, "n_docs_with_agent_head": int(n_head),
       "world": {s: {w: {c: float(v) for c, v in r.items()} for w, r in t.iterrows()} for s, t in W.items()},
       "lang": {s: {w: {c: (float(v) if not isinstance(v, str) else v) for c, v in r.items()} for w, r in t.iterrows()} for s, t in L.items()},
       "lang_specific": spec.to_dict(orient="records"), "binomial_sig": sigB.to_dict(orient="records"), "llm_family_union": {"world": llm_union["world"], "world_pct": round(100 * p0, 2), **llm_union_stats}}
p = VER / f"agent_colloc_{TODAY}.json"; p.write_text(json.dumps(out, ensure_ascii=False, indent=1)); print("saved", p)